# Model Training File

In [36]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor 
import warnings

In [37]:
df = pd.read_csv('data/students_data.csv')

In [38]:
df.head()

,Gender,Race_Ethnicity,Parental_Education,Lunch,Test_Preparation_Course,Math_Score,Reading_Score,Writing_Score
0,Male,Group C,Associate's Degree,Free,Not Completed,58,58,61
1,Female,Group C,Some College,Paid,Not Completed,40,43,95
2,Male,Group C,College,Paid,Completed,88,85,58
3,Male,Group D,Master's Degree,Paid,Not Completed,99,42,45
4,Male,Group D,Some College,Paid,Not Completed,87,67,97


In [39]:
X = df.drop(columns=['Math_Score'])
y = df['Math_Score']

In [40]:
# create a column transformer with 3 types of transformers
num_features = X.select_dtypes(exclude="str").columns
cat_features = X.select_dtypes(include="str").columns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numerical_transformer = StandardScaler()
oh_transformer = OneHotEncoder()

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, cat_features),
        ("StandardScaler", numerical_transformer, num_features),
    ]
)



In [41]:
X = preprocessor.fit_transform(X)

In [42]:
X.shape

(1000, 18)

In [43]:
from sklearn.model_selection import train_test_split

In [44]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

((800, 18), (200, 18))

### Create an Evaluate Function to give all metrices after model training 

In [45]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mse)
    r2_square = r2_score(true, predicted)
    return mae, mse, rmse, r2_square

In [46]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge

In [47]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbour Regression": KNeighborsRegressor(),
    "DecisionTreeRegressor": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(),
    "CatBoosting Regressor": CatBoostRegressor(),
    "AdaBoostRegressor": AdaBoostRegressor() 
}

model_list = []
r2_list = []

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae, model_train_mse, model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)
    model_test_mae, model_test_mse, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)


    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])

    print("Model Performance for Training Set")
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    r2_list.append(model_test_r2)
    
    print('='*35)
    print('\n')


Linear Regression
Model Performance for Training Set
- Root Mean Squared Error: 17.8377
- Mean Absolute Error: 15.6307
- R2 Score: 0.0100
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 17.7674
- Mean Absolute Error: 15.2190
- R2 Score: 0.0045


Lasso
Model Performance for Training Set
- Root Mean Squared Error: 17.9278
- Mean Absolute Error: 15.7718
- R2 Score: 0.0000
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 17.8173
- Mean Absolute Error: 15.2805
- R2 Score: -0.0011


Ridge
Model Performance for Training Set
- Root Mean Squared Error: 17.8377
- Mean Absolute Error: 15.6313
- R2 Score: 0.0100
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 17.7674
- Mean Absolute Error: 15.2191
- R2 Score: 0.0045


K-Neighbour Regression
Model Performance for Training Set
- Root Mean Squared Error: 16.3903
- Mean Absolute Error: 14.0573
- R2 Score: 0.1642
-------

In [48]:
pd.DataFrame(list(zip(model_list, r2_list)), columns=['Model Name', 'R2_Score']).sort_values(by=["R2_Score"],ascending=False)

,Model Name,R2_Score
2,Ridge,0.004547
0,Linear Regression,0.004546
8,AdaBoostRegressor,-0.000380
1,Lasso,-0.001061
5,Random Forest Regressor,-0.072495
3,K-Neighbour Regression,-0.142687
7,CatBoosting Regressor,-0.210665
6,XGBRegressor,-0.427360
4,DecisionTreeRegressor,-1.165971
